# arynews.tv Scraper — 20-Session Parallel (v8)

**Workflow**:
1. Set `CHROME_NUMBER` in Cell 1 (1-20)
2. Run all cells
3. Each chrome scrapes its unique 1/20 slice of URLs (no overlap)
4. At the end, **auto-downloads** your chrome's data file to your PC
5. After all 20 chromes done, run Cell 7 once to merge everything

## Time Estimate (500K articles, 8 art/s per chrome)
- 1 session: ~17 hours
- 10 sessions: ~1.7 hours
- **20 sessions: ~1 hour**

## Files Used
- **Input**: `MyDrive/urdu_corpus/arynews/urls/all_urls.jsonl` (you already have this)
- **Output**: `MyDrive/urdu_corpus/arynews/articles/articles_chromeN_XXXX.jsonl.gz`
- **Downloaded to PC**: `arynews_chromeN.zip` (your chunk's articles)

## Cell 1 — SET YOUR CHROME NUMBER ⚠️

**Each parallel Colab session must have a UNIQUE `CHROME_NUMBER` from 1 to 20.**

| Session | CHROME_NUMBER |
|---------|---------------|
| Tab 1 | `1` |
| Tab 2 | `2` |
| ... | ... |
| Tab 20 | `20` |

Set `CHROME_NUMBER = 0` to process ALL URLs in one session (slow).

In [ ]:
# ============================================================
# ⚠️ SET THIS FOR EACH PARALLEL SESSION (1-20)
CHROME_NUMBER = 1
TOTAL_CHROMES = 20
# ============================================================
print(f'This session: CHROME_NUMBER={CHROME_NUMBER} of {TOTAL_CHROMES}')
if CHROME_NUMBER == 0:
    print('Processing ALL URLs in single session (slow, ~17h)')
else:
    print(f'Will process chunk {CHROME_NUMBER}/{TOTAL_CHROMES} = ~1/{TOTAL_CHROMES} of all URLs')

## Cell 2 — Install Dependencies

In [ ]:
!pip install -q aiohttp lxml tqdm
print('Dependencies installed.')

## Cell 3 — Mount Google Drive

This connects to your Drive where `all_urls.jsonl` is already stored.

Your URLs file should be at: `MyDrive/urdu_corpus/arynews/urls/all_urls.jsonl`

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_BASE = '/content/drive/MyDrive/urdu_corpus/arynews'
URLS_FILE = f'{DRIVE_BASE}/urls/all_urls.jsonl'

# Make sure output dirs exist
for sub in ['urls', 'articles', 'logs', 'checkpoint']:
    os.makedirs(f'{DRIVE_BASE}/{sub}', exist_ok=True)

if os.path.exists(URLS_FILE):
    with open(URLS_FILE) as f:
        n_urls = sum(1 for line in f if line.strip())
    print(f'Drive mounted ✓')
    print(f'URLs file found: {URLS_FILE}')
    print(f'Total URLs available: {n_urls:,}')
    print(f'Your chunk (chrome {CHROME_NUMBER}): ~{n_urls // TOTAL_CHROMES:,} URLs')
else:
    print(f'Drive mounted ✓')
    print(f'⚠ URLs file NOT FOUND at: {URLS_FILE}')
    print(f'  Cell 4 will run sitemap discovery to create it (~5-10 min)')

## Cell 4 — Scraper Code

This cell loads all functions. It will:
1. Load `all_urls.jsonl` from your Drive (or run discovery if not found)
2. Take this chrome's unique slice of URLs
3. Scrape with 80 concurrent async connections

In [ ]:
import os, sys, json, gzip, time, random, logging, asyncio, threading, re
from pathlib import Path
from datetime import datetime, timezone
from urllib.parse import urlparse
from collections import Counter

import aiohttp
import lxml.html
import xml.etree.ElementTree as ET
from tqdm import tqdm

# === CONFIG ===
BASE_URL = 'https://urdu.arynews.tv'
DRIVE_BASE = '/content/drive/MyDrive/urdu_corpus/arynews'
URLS_FILE = f'{DRIVE_BASE}/urls/all_urls.jsonl'
ARTICLES_DIR = Path(f'{DRIVE_BASE}/articles')
LOGS_DIR = Path(f'{DRIVE_BASE}/logs')
CHECKPOINT_DIR = Path(f'{DRIVE_BASE}/checkpoint')

CONCURRENCY = 80
REQUEST_TIMEOUT = 15
MAX_RETRIES = 2
RETRY_DELAY = 2

HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
    'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8',
    'Accept-Language': 'en-US,en;q=0.5',
    'Accept-Encoding': 'gzip, deflate',
    'Connection': 'keep-alive',
    'Upgrade-Insecure-Requests': '1',
    'Sec-Fetch-Dest': 'document',
    'Sec-Fetch-Mode': 'navigate',
    'Sec-Fetch-Site': 'none',
    'Sec-Fetch-User': '?1',
    'Cache-Control': 'max-age=0',
}

FAILURE_THRESHOLD = 10
FAILURE_WINDOW = 30
CONCURRENCY_REDUCE_FACTOR = 0.5
CONCURRENCY_MIN = 10
CHECKPOINT_INTERVAL_SECONDS = 60
PAGES_PER_SHARD = 500

SUFFIX = f'chrome{CHROME_NUMBER}' if CHROME_NUMBER > 0 else 'all'

# === LOGGING ===
def setup_logging():
    LOGS_DIR.mkdir(parents=True, exist_ok=True)
    log_file = LOGS_DIR / f'scrape_{SUFFIX}_{datetime.now().strftime("%Y%m%d_%H%M%S")}.log'
    root = logging.getLogger()
    for h in list(root.handlers): root.removeHandler(h)
    logging.basicConfig(
        level=logging.INFO,
        format=f'%(asctime)s [{SUFFIX}] [%(levelname)s] %(message)s',
        handlers=[logging.FileHandler(log_file, encoding='utf-8'), logging.StreamHandler()],
    )

log = logging.getLogger('arynews')

# === LOAD URLS ===
def load_all_urls():
    if os.path.exists(URLS_FILE):
        log.info(f'Loading URLs from {URLS_FILE}')
        urls = []
        with open(URLS_FILE, encoding='utf-8') as f:
            for line in f:
                if line.strip():
                    urls.append(json.loads(line))
        log.info(f'Loaded {len(urls):,} URLs from cache')
        return urls
    
    # Not found - run discovery
    log.info('URLs file not found. Running sitemap discovery...')
    urls = run_discovery()
    return urls

def is_article_url(url):
    if not url.startswith(BASE_URL + '/'):
        return False
    if any(x in url for x in ['/category/', '/page/', '/tag/', '/author/', '?s=', 'wp-']):
        return False
    path = urlparse(url).path.strip('/')
    return '/' not in path and len(path) > 5

SM_NS = {'sm': 'http://www.sitemaps.org/schemas/sitemap/0.9'}

async def fetch_sitemap_async(session, url, sem):
    async with sem:
        for attempt in range(3):
            try:
                async with session.get(url, headers=HEADERS, timeout=aiohttp.ClientTimeout(total=30)) as r:
                    if r.status == 200:
                        return await r.read()
                    elif r.status in (429, 503):
                        await asyncio.sleep(5)
                    else:
                        return None
            except Exception:
                await asyncio.sleep(2)
        return None

def run_discovery():
    """Run sitemap discovery if URLs file doesn't exist."""
    async def _discover():
        log.info('[DISCOVERY] Fetching sitemap index...')
        sem = asyncio.Semaphore(8)
        connector = aiohttp.TCPConnector(limit=10, keepalive_timeout=60)
        async with aiohttp.ClientSession(connector=connector) as session:
            index_data = await fetch_sitemap_async(session, f'{BASE_URL}/sitemap.xml', sem)
            if index_data is None:
                log.error('[DISCOVERY] Failed to fetch sitemap index!')
                return []
            root = ET.fromstring(index_data)
            sitemap_urls = []
            for sm in root.findall('sm:sitemap', SM_NS):
                loc = sm.find('sm:loc', SM_NS)
                if loc is not None and loc.text:
                    sitemap_urls.append(loc.text)
            post_sitemaps = [u for u in sitemap_urls if 'post-sitemap' in u]
            log.info(f'[DISCOVERY] {len(post_sitemaps)} post-sitemaps to fetch')
            
            all_urls = set()
            BATCH = 20
            for batch_start in range(0, len(post_sitemaps), BATCH):
                batch = post_sitemaps[batch_start:batch_start + BATCH]
                tasks = [fetch_sitemap_async(session, u, sem) for u in batch]
                results = await asyncio.gather(*tasks)
                for data in results:
                    if data is None:
                        continue
                    try:
                        sm_root = ET.fromstring(data)
                        for u in sm_root.findall('sm:url', SM_NS):
                            loc = u.find('sm:loc', SM_NS)
                            if loc is not None and loc.text and is_article_url(loc.text):
                                all_urls.add(loc.text)
                    except Exception:
                        continue
                done = min(batch_start + BATCH, len(post_sitemaps))
                log.info(f'[DISCOVERY] {done}/{len(post_sitemaps)} sitemaps: {len(all_urls):,} URLs')
        
        return sorted(all_urls)
    
    urls_list = asyncio.run(_discover())
    if not urls_list:
        return []
    
    # Save for future use
    os.makedirs(os.path.dirname(URLS_FILE), exist_ok=True)
    with open(URLS_FILE, 'w', encoding='utf-8') as f:
        for u in urls_list:
            f.write(json.dumps({'url': u}, ensure_ascii=False) + '\n')
    log.info(f'[DISCOVERY] Saved {len(urls_list):,} URLs to {URLS_FILE}')
    return [{'url': u} for u in urls_list]

def load_my_chunk(all_urls):
    """Take this chrome's unique slice of URLs."""
    if CHROME_NUMBER == 0:
        log.info(f'CHROME_NUMBER=0: processing ALL {len(all_urls):,} URLs')
        return all_urls
    
    chunk_size = len(all_urls) // TOTAL_CHROMES
    start = (CHROME_NUMBER - 1) * chunk_size
    if CHROME_NUMBER == TOTAL_CHROMES:
        end = len(all_urls)  # last chrome takes remainder
    else:
        end = CHROME_NUMBER * chunk_size
    
    my_chunk = all_urls[start:end]
    log.info(f'CHROME_NUMBER={CHROME_NUMBER}/{TOTAL_CHROMES}: URLs [{start:,}:{end:,}] = {len(my_chunk):,} URLs')
    return my_chunk

# === EXTRACTION ===
_RE_NEWLINES = re.compile(r'\n\s*\n+')
_RE_SPACES = re.compile(r'[ \t]+')

def extract_article(html, url):
    if not html:
        return None
    try:
        tree = lxml.html.fromstring(html)
    except Exception:
        return None
    title_nodes = tree.xpath('//h1[contains(@class, "tdb-title-text")]/text()')
    if not title_nodes:
        title_nodes = tree.xpath('//article//h1//text()')
    if not title_nodes:
        return None
    title = title_nodes[0].strip() if isinstance(title_nodes[0], str) else ''.join(title_nodes).strip()
    if not title:
        return None
    body_nodes = tree.xpath('//div[contains(@class, "td-post-content")]')
    if not body_nodes:
        return None
    body_elem = body_nodes[0]
    for junk_xpath in ['.//style', './/script',
                       './/div[contains(@class, "code-block")]',
                       './/div[contains(@class, "td_block_template")]',
                       './/div[contains(@class, "related")]',
                       './/div[contains(@class, "share")]',
                       './/div[contains(@class, "wp-post-navigation")]',
                       './/div[contains(@class, "td-post-source-tags")]',
                       './/div[contains(@class, "td-post-sharing")]']:
        for junk in body_elem.xpath(junk_xpath):
            if junk.getparent() is not None:
                junk.getparent().remove(junk)
    body = body_elem.text_content().strip()
    body = _RE_NEWLINES.sub('\n\n', body)
    body = _RE_SPACES.sub(' ', body)
    if not body or len(body) < 100:
        return None
    date_nodes = tree.xpath('//time[contains(@class, "entry-date")]/@datetime')
    if not date_nodes:
        date_nodes = tree.xpath('//time/@datetime')
    published = date_nodes[0] if date_nodes else None
    cat_nodes = tree.xpath('//a[contains(@href, "/category/") and not(contains(@href, "/page/"))]/text()')
    category = 'unknown'
    for c in cat_nodes:
        c = c.strip() if isinstance(c, str) else ''
        if c and len(c) < 30 and c not in ['صفحہ اول', 'ہوم']:
            category = c
            break
    return {'url': url, 'title': title, 'category': category,
            'published_date': published, 'body_text': body,
            'char_count': len(body),
            'scraped_at': datetime.now(timezone.utc).isoformat()}

# === CHECKPOINT ===
def get_checkpoint_file():
    return CHECKPOINT_DIR / f'progress_{SUFFIX}.json'

def load_checkpoint():
    cp_file = get_checkpoint_file()
    if cp_file.exists():
        with open(cp_file, encoding='utf-8') as f:
            data = json.load(f)
        return {'completed_urls': set(data.get('completed_urls', [])),
                'failed_urls': set(data.get('failed_urls', [])),
                'total_saved': data.get('total_saved', 0),
                'current_shard_idx': data.get('current_shard_idx', 0),
                'current_shard_count': data.get('current_shard_count', 0),
                'start_time': data.get('start_time', datetime.now(timezone.utc).isoformat())}
    return {'completed_urls': set(), 'failed_urls': set(), 'total_saved': 0,
            'current_shard_idx': 0, 'current_shard_count': 0,
            'start_time': datetime.now(timezone.utc).isoformat()}

def save_checkpoint(state):
    CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
    cp_file = get_checkpoint_file()
    data = {'completed_urls': list(state['completed_urls']),
            'failed_urls': list(state['failed_urls']),
            'total_saved': state['total_saved'],
            'current_shard_idx': state['current_shard_idx'],
            'current_shard_count': state['current_shard_count'],
            'start_time': state['start_time'],
            'last_saved_at': datetime.now(timezone.utc).isoformat()}
    tmp = cp_file.with_suffix('.tmp')
    with open(tmp, 'w', encoding='utf-8') as f:
        json.dump(data, f, ensure_ascii=False)
    tmp.replace(cp_file)

def get_current_shard_path(state):
    return ARTICLES_DIR / f'articles_{SUFFIX}_{state["current_shard_idx"]:04d}.jsonl.gz'

_file_lock = threading.Lock()
def append_to_shard_sync(state, article):
    with _file_lock:
        ARTICLES_DIR.mkdir(parents=True, exist_ok=True)
        shard_path = get_current_shard_path(state)
        with gzip.open(shard_path, 'at', encoding='utf-8') as f:
            f.write(json.dumps(article, ensure_ascii=False) + '\n')
        state['current_shard_count'] += 1
        state['total_saved'] += 1
        if state['current_shard_count'] >= PAGES_PER_SHARD:
            state['current_shard_idx'] += 1
            state['current_shard_count'] = 0

# === ASYNC FETCH ===
async def fetch_one(session, url, sem):
    async with sem:
        for attempt in range(MAX_RETRIES + 1):
            try:
                async with session.get(url, headers=HEADERS,
                                       timeout=aiohttp.ClientTimeout(total=REQUEST_TIMEOUT)) as r:
                    if r.status == 200:
                        html = await r.text()
                        article = await asyncio.get_event_loop().run_in_executor(
                            None, extract_article, html, url)
                        return url, article
                    elif r.status in (429, 503):
                        await asyncio.sleep(RETRY_DELAY * 2)
                        continue
                    else:
                        return url, None
            except (asyncio.TimeoutError, aiohttp.ClientError):
                if attempt < MAX_RETRIES:
                    await asyncio.sleep(RETRY_DELAY)
                else:
                    return url, None
            except Exception:
                return url, None
    return url, None

async def fetch_all_articles(url_records):
    state = load_checkpoint()
    log.info(f'Resuming: {len(state["completed_urls"]):,} done')
    pending = [r for r in url_records if r['url'] not in state['completed_urls']]
    log.info(f'Pending: {len(pending):,} articles')
    if not pending:
        log.info('Nothing to do.')
        return state
    
    current_concurrency = CONCURRENCY
    sem = asyncio.Semaphore(current_concurrency)
    failure_timestamps = []
    last_checkpoint = time.time()
    fetch_start = time.time()
    
    connector = aiohttp.TCPConnector(limit=current_concurrency + 20,
                                     limit_per_host=current_concurrency,
                                     keepalive_timeout=30)
    
    pbar = tqdm(total=len(pending), desc=f'Chrome{CHROME_NUMBER}', unit='art')
    
    async with aiohttp.ClientSession(connector=connector) as session:
        BATCH = 200
        for batch_start in range(0, len(pending), BATCH):
            batch = pending[batch_start:batch_start + BATCH]
            tasks = [fetch_one(session, r['url'], sem) for r in batch]
            
            for coro in asyncio.as_completed(tasks):
                url, article = await coro
                if article is not None:
                    await asyncio.get_event_loop().run_in_executor(
                        None, append_to_shard_sync, state, article)
                    state['completed_urls'].add(url)
                else:
                    state['failed_urls'].add(url)
                    failure_timestamps.append(time.time())
                pbar.update(1)
                pbar.set_postfix(saved=state['total_saved'],
                                 failed=len(state['failed_urls']),
                                 conc=current_concurrency)
            
            cutoff = time.time() - FAILURE_WINDOW
            failure_timestamps[:] = [t for t in failure_timestamps if t > cutoff]
            
            if len(failure_timestamps) >= FAILURE_THRESHOLD:
                new_c = max(CONCURRENCY_MIN, int(current_concurrency * CONCURRENCY_REDUCE_FACTOR))
                if new_c != current_concurrency:
                    log.warning(f'[ADAPTIVE] {len(failure_timestamps)} failures — reduce {current_concurrency} → {new_c}')
                    current_concurrency = new_c
                    sem = asyncio.Semaphore(current_concurrency)
                    failure_timestamps.clear()
                    await asyncio.sleep(2)
            
            if time.time() - last_checkpoint > CHECKPOINT_INTERVAL_SECONDS:
                await asyncio.get_event_loop().run_in_executor(None, save_checkpoint, state)
                last_checkpoint = time.time()
    
    pbar.close()
    save_checkpoint(state)
    elapsed = time.time() - fetch_start
    rate = state['total_saved'] / max(elapsed, 1)
    log.info(f'DONE. {state["total_saved"]:,} saved, {len(state["failed_urls"]):,} failed in {elapsed:.0f}s ({rate:.1f} art/s)')
    return state

# === MAIN ===
def main():
    for d in [ARTICLES_DIR, LOGS_DIR, CHECKPOINT_DIR]:
        d.mkdir(parents=True, exist_ok=True)
    setup_logging()
    log.info('=' * 60)
    log.info(f'ARYNEWS ASYNC SCRAPER — CHROME {CHROME_NUMBER}/{TOTAL_CHROMES}')
    log.info(f'Concurrency: {CONCURRENCY}, Timeout: {REQUEST_TIMEOUT}s')
    log.info('=' * 60)
    
    all_urls = load_all_urls()
    if not all_urls:
        log.error('No URLs to process!')
        return
    
    my_chunk = load_my_chunk(all_urls)
    state = asyncio.run(fetch_all_articles(my_chunk))
    
    log.info('=' * 60)
    log.info(f'CHROME {CHROME_NUMBER} COMPLETE')
    log.info(f'Total saved: {state["total_saved"]:,}')
    log.info(f'Failed: {len(state["failed_urls"]):,}')
    log.info('=' * 60)

print(f'Scraper loaded. CHROME_NUMBER={CHROME_NUMBER}/{TOTAL_CHROMES}, CONCURRENCY={CONCURRENCY}')
print(f'URLs file: {URLS_FILE}')
print(f'Output suffix: {SUFFIX}')
print('Run the next cell to start scraping.')

## Cell 5 — Start Scraping

Run this cell to start scraping your chunk.

**For 20 parallel sessions**: Open this notebook in 20 Colab sessions, set `CHROME_NUMBER = 1` to `20` in each, run all cells.

**If disconnected**: re-run Cells 1-5. Loads checkpoint and resumes.

In [ ]:
main()

# Post-scrape summary
import gzip, json
from pathlib import Path

shards = sorted(ARTICLES_DIR.glob(f'articles_{SUFFIX}_*.jsonl.gz'))
total_size = sum(s.stat().st_size for s in shards) / 1024 / 1024
print(f'\n=== CHROME {CHROME_NUMBER} OUTPUT ===')
print(f'Shards: {len(shards)}, Total size: {total_size:.1f} MB')
for s in shards[-5:]:
    print(f'  {s.name}: {s.stat().st_size/1024/1024:.2f} MB')

if shards:
    print(f'\nSample articles from latest shard:')
    with gzip.open(shards[-1], 'rt', encoding='utf-8') as f:
        for i, line in enumerate(f):
            if i >= 3: break
            art = json.loads(line)
            print(f'\n  [{i+1}] {art["title"][:80]}')
            print(f'      Cat: {art["category"]} | Date: {art["published_date"]} | Len: {art["char_count"]}')
            print(f'      Body: {art["body_text"][:150]}...')

## Cell 6 — Download This Chrome's Data to Your PC

After scraping finishes, run this cell. It will:
1. Combine all your chrome's shard files into one `.jsonl.gz` file
2. **Auto-download** it to your PC as `arynews_chromeN.jsonl.gz`

If the download doesn't start automatically, look for the download icon in the cell output.

In [ ]:
import gzip, json, shutil
from pathlib import Path
from google.colab import files

# Combine all this chrome's shards into one file
shards = sorted(ARTICLES_DIR.glob(f'articles_{SUFFIX}_*.jsonl.gz'))
print(f'Found {len(shards)} shard files for {SUFFIX}')

if not shards:
    print('No shards to download!')
else:
    # Combine into single file
    output_filename = f'arynews_{SUFFIX}.jsonl.gz'
    output_path = Path('/content') / output_filename
    
    total_articles = 0
    with gzip.open(output_path, 'wt', encoding='utf-8') as out:
        for shard in shards:
            print(f'  Reading {shard.name}...')
            with gzip.open(shard, 'rt', encoding='utf-8') as f:
                for line in f:
                    out.write(line)
                    total_articles += 1
    
    size_mb = output_path.stat().st_size / 1024 / 1024
    print(f'\nCombined: {output_filename}')
    print(f'  Articles: {total_articles:,}')
    print(f'  Size: {size_mb:.1f} MB')
    print(f'\nDownloading to your PC...')
    
    # Trigger browser download
    files.download(str(output_path))
    print(f'\n✓ Download started. Check your browser downloads folder.')
    print(f'  File: {output_filename}')
    print(f'  Size: {size_mb:.1f} MB')

## Cell 7 — Check Progress of All Chromes (optional)

Run this in ANY session to see how all 20 chromes are doing.

In [ ]:
import json
from pathlib import Path

checkpoint_dir = Path(f'{DRIVE_BASE}/checkpoint')
articles_dir = Path(f'{DRIVE_BASE}/articles')

print('=== PROGRESS ACROSS ALL CHROMES ===')
print(f'{"Chrome":<10} {"Saved":>10} {"Failed":>10} {"Last Saved":<25}')
print('-' * 60)

total_saved = 0
total_failed = 0
for cp_file in sorted(checkpoint_dir.glob('progress_chrome*.json')):
    with open(cp_file) as f:
        data = json.load(f)
    chrome = cp_file.stem.replace('progress_chrome', '')
    saved = data.get('total_saved', 0)
    failed = len(data.get('failed_urls', []))
    last = data.get('last_saved_at', 'n/a')[:19]
    total_saved += saved
    total_failed += failed
    print(f'{chrome:<10} {saved:>10,} {failed:>10,} {last:<25}')

print('-' * 60)
print(f'{"TOTAL":<10} {total_saved:>10,} {total_failed:>10,}')

all_shards = list(articles_dir.glob('articles_chrome*_*.jsonl.gz'))
total_size = sum(s.stat().st_size for s in all_shards) / 1024 / 1024
print(f'\nTotal shard files: {len(all_shards)}, Total size: {total_size:.1f} MB')

## Cell 8 — Merge All Chromes (run ONCE after all 20 finish)

After all 20 chromes complete, run this ONCE to:
1. Merge all `articles_chromeN_*.jsonl.gz` files into one
2. Remove duplicates
3. **Download** the final merged file `arynews_merged.jsonl.gz` to your PC

This is the final dataset for arynews.tv — ready for the next site!

In [ ]:
import gzip, json
from pathlib import Path
from collections import defaultdict
from google.colab import files

articles_dir = Path(f'{DRIVE_BASE}/articles')
output_file = articles_dir / 'arynews_merged.jsonl.gz'

all_shards = sorted(articles_dir.glob('articles_chrome*_*.jsonl.gz'))
all_shards += sorted(articles_dir.glob('articles_all_*.jsonl.gz'))
print(f'Found {len(all_shards)} shard files')

if not all_shards:
    print('No shards found! Run scraping first.')
else:
    seen_urls = set()
    total = 0
    dupes = 0
    cat_counts = defaultdict(int)
    
    with gzip.open(output_file, 'wt', encoding='utf-8') as out:
        for shard in all_shards:
            print(f'  Reading {shard.name}...')
            with gzip.open(shard, 'rt', encoding='utf-8') as f:
                for line in f:
                    art = json.loads(line)
                    if art['url'] in seen_urls:
                        dupes += 1
                        continue
                    seen_urls.add(art['url'])
                    out.write(line)
                    total += 1
                    cat_counts[art.get('category', 'unknown')] += 1
    
    size_mb = output_file.stat().st_size / 1024 / 1024
    print(f'\n=== MERGE COMPLETE ===')
    print(f'Total unique articles: {total:,}')
    print(f'Duplicates removed: {dupes:,}')
    print(f'Output: {output_file}')
    print(f'Size: {size_mb:.1f} MB')
    print(f'\nArticles per category:')
    for cat, count in sorted(cat_counts.items(), key=lambda x: -x[1]):
        print(f'  {cat}: {count:,}')
    
    # Download the merged file to PC
    print(f'\nDownloading merged file to your PC...')
    files.download(str(output_file))
    print(f'\n✓ Download started: arynews_merged.jsonl.gz ({size_mb:.1f} MB)')